<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip install -q duckdb huggingface_hub pandas numpy

In [11]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

# Load Hugging Face token safely
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it in the Colab Secrets panel."
    )

user_info = whoami(token=HF_TOKEN)
print("Hugging Face login successful:", user_info["name"])

# Connect DuckDB
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection ready.")
print("Development month: March 2026")
print("Feature window: March 1–15")
print("Outcome window: March 16–31")

Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026
Feature window: March 1–15
Outcome window: March 16–31


## Audit Frame

The source table contains one row per client, content page, and day.

For this signal audit, I aggregate the daily records into one row per anonymized content page.

The feature window is March 1–15, 2026. The outcome window is March 16–31, 2026.

All tested signals are calculated only from the feature window. The outcome window is used only to evaluate whether each signal is associated with the provisional declining proxy.

The declining proxy equals 1 when the page's average daily impressions in the outcome window are more than 20% below its average daily impressions in the feature window.

This proxy measures observed movement. It does not prove that a page needs a refresh or that editing it would cause recovery.

In [12]:
audit_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature-window impressions
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feature_impressions,

        -- Feature-window clicks
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feature_clicks,

        -- Feature-window average position
        AVG(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        -- Feature-window position volatility
        STDDEV_SAMP(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        -- Feature-window active days
        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END
        ) AS feature_active_days,

        -- Feature-window available days
        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01'
                     AND DATE '2026-03-15'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS feature_available_days,

        -- Outcome-window impressions
        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS outcome_impressions,

        -- Outcome-window available days
        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-16'
                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {MARCH_FACT}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Raw page-level rows:", f"{len(audit_frame):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw page-level rows: 331,437


In [13]:
# Keep pages with enough evidence
audit_frame = audit_frame[
    (audit_frame["feature_impressions"] >= 100)
    & (audit_frame["feature_available_days"] >= 5)
    & (audit_frame["outcome_available_days"] >= 5)
].copy()

# Daily averages make the two windows comparable
audit_frame["feature_daily_impressions"] = (
    audit_frame["feature_impressions"]
    / audit_frame["feature_available_days"]
)

audit_frame["outcome_daily_impressions"] = (
    audit_frame["outcome_impressions"]
    / audit_frame["outcome_available_days"]
)

# Provisional outcome proxy
audit_frame["is_declining_proxy"] = (
    audit_frame["outcome_daily_impressions"]
    < 0.80 * audit_frame["feature_daily_impressions"]
).astype(int)

# Safe feature-window CTR
audit_frame["feature_ctr"] = np.where(
    audit_frame["feature_impressions"] > 0,
    audit_frame["feature_clicks"]
    / audit_frame["feature_impressions"],
    np.nan,
)

# Fill volatility when too few valid observations exist
volatility_fill = audit_frame[
    "feature_position_volatility"
].median()

audit_frame["feature_position_volatility"] = (
    audit_frame["feature_position_volatility"]
    .fillna(volatility_fill)
)

print("Eligible audit rows:", f"{len(audit_frame):,}")
print(
    "Declining proxy rate:",
    round(audit_frame["is_declining_proxy"].mean(), 3),
)

audit_frame.head()

Eligible audit rows: 76,201
Declining proxy rate: 0.345


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_position_volatility,feature_active_days,feature_available_days,outcome_impressions,outcome_available_days,feature_daily_impressions,outcome_daily_impressions,is_declining_proxy,feature_ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,1.020755,15,15,2350.0,16,278.200000,146.8750,1,0.001438
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,2.925536,15,15,208.0,16,16.333333,13.0000,1,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1.324947,15,15,1925.0,16,247.000000,120.3125,1,0.000810
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,0.777470,15,15,2504.0,16,162.666667,156.5000,0,0.003279
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,2.113680,15,15,189.0,16,16.000000,11.8125,1,0.004167


# 1. Distributions

Before choosing thresholds, I inspected the distributions of the main signals.

The inspected fields are:

- feature-window impressions;
- feature-window clicks;
- feature-window CTR;
- average position;
- position volatility.

Impressions and clicks may have heavy tails. A heavy tail means most pages have modest values while a small group has extremely large values.

This matters because a raw linear score could be dominated by a few extremely large pages. Bucketed or transformed scores may therefore be more appropriate for the later baseline.

In [14]:
distribution_columns = [
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_volatility",
]

distribution_summary = (
    audit_frame[distribution_columns]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)

distribution_summary

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
feature_impressions,76201.0,1642.757077,3603.414164,100.000000,243.000000,590.000000,1570.000000,3816.000000,6410.000000,16344.000000,161575.000000
feature_clicks,76201.0,4.945408,18.371146,0.000000,0.000000,1.000000,4.000000,12.000000,21.000000,60.000000,2395.000000
feature_ctr,76201.0,0.002887,0.004640,0.000000,0.000000,0.001346,0.004172,0.007722,0.010695,0.020147,0.213592
feature_avg_position,76201.0,12.589168,12.561123,0.012229,4.366972,7.588764,16.473467,30.042364,38.392042,61.296403,127.620709
feature_position_volatility,76201.0,4.897639,4.705462,0.020824,1.501515,3.403126,6.636631,11.109750,14.330018,21.731766,130.995590


In [15]:
heavy_tail_receipt = pd.DataFrame({
    "field": distribution_columns,
    "median": [
        audit_frame[column].median()
        for column in distribution_columns
    ],
    "p95": [
        audit_frame[column].quantile(0.95)
        for column in distribution_columns
    ],
    "p99": [
        audit_frame[column].quantile(0.99)
        for column in distribution_columns
    ],
    "maximum": [
        audit_frame[column].max()
        for column in distribution_columns
    ],
})

heavy_tail_receipt["p99_to_median_ratio"] = (
    heavy_tail_receipt["p99"]
    / heavy_tail_receipt["median"].replace(0, np.nan)
)

heavy_tail_receipt

,field,median,p95,p99,maximum,p99_to_median_ratio
0,feature_impressions,590.000000,6410.000000,16344.000000,161575.000000,27.701695
1,feature_clicks,1.000000,21.000000,60.000000,2395.000000,60.000000
2,feature_ctr,0.001346,0.010695,0.020147,0.213592,14.968864
3,feature_avg_position,7.588764,38.392042,61.296403,127.620709,8.077258
4,feature_position_volatility,3.403126,14.330018,21.731766,130.995590,6.385825


### Distribution Observation

The distribution table should be interpreted using the median, 95th percentile, 99th percentile, and maximum.

A large difference between the median and upper percentiles indicates a heavy tail.

The later baseline should therefore avoid allowing a small number of extremely high-volume pages to dominate the entire score. Percentile ranks, logarithmic transformations, or capped bucket scores are safer than using raw impressions directly.

## Signal Test 1 — Visibility

### Hypothesis

Pages with more feature-window impressions represent higher-value review opportunities because more search exposure is at stake.

This test does not assume that high visibility automatically causes decline. It checks whether decline rates differ across visibility buckets and whether volume is useful as prioritization context.

### Verdict options

- CONFIRMED
- OPPOSITE
- MIXED
- FALSE

In [16]:
audit_frame["visibility_bucket"] = pd.cut(
    audit_frame["feature_impressions"],
    bins=[
        99,
        499,
        1_999,
        9_999,
        np.inf,
    ],
    labels=[
        "100-499",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
    include_lowest=True,
)

signal_1_table = (
    audit_frame
    .groupby(
        "visibility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
    )
    .reset_index()
)

signal_1_table["decline_rate_pct"] = (
    100 * signal_1_table["decline_rate"]
)

signal_1_table

,visibility_bucket,n,declining_pages,decline_rate,median_impressions,decline_rate_pct
0,100-499,34590,12062,0.348714,224.0,34.871350
1,"500-1,999",26261,8784,0.334488,922.0,33.448840
2,"2,000-9,999",13474,4670,0.346593,3443.5,34.659344
3,"10,000+",1876,748,0.398721,14786.5,39.872068


In [17]:
visibility_rates = (
    signal_1_table["decline_rate"]
    .dropna()
    .to_numpy()
)

visibility_range = (
    visibility_rates.max()
    - visibility_rates.min()
)

if visibility_range < 0.03:
    signal_1_verdict = "FALSE"
elif np.all(np.diff(visibility_rates) >= 0):
    signal_1_verdict = "CONFIRMED"
elif np.all(np.diff(visibility_rates) <= 0):
    signal_1_verdict = "OPPOSITE"
else:
    signal_1_verdict = "MIXED"

print("Signal 1 verdict:", signal_1_verdict)
print(
    "Decline-rate range:",
    round(float(visibility_range), 3),
)

Signal 1 verdict: MIXED
Decline-rate range: 0.064


### Signal 1 Verdict

**Verdict: Replace this line with the printed verdict.**

Interpretation:

Visibility should mainly represent potential business impact, not proof that a page requires a refresh.

If the decline rates do not move consistently across buckets, volume should be used as a prioritization weight rather than as a stand-alone decline rule.

## Signal Test 2 — Position Volatility

### Hypothesis

Pages with more unstable feature-window positions may show a higher later decline rate.

Position volatility is measured only during March 1–15, so it is available before the outcome window.

A positive result would support using ranking instability as one part of a review-priority rule. It would not prove why the ranking moved.

In [18]:
# Use rank-based quartiles so each bucket has meaningful sample size
audit_frame["volatility_bucket"] = pd.qcut(
    audit_frame["feature_position_volatility"].rank(
        method="first"
    ),
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
    ],
)

signal_2_table = (
    audit_frame
    .groupby(
        "volatility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_volatility=(
            "feature_position_volatility",
            "median",
        ),
    )
    .reset_index()
)

signal_2_table["decline_rate_pct"] = (
    100 * signal_2_table["decline_rate"]
)

signal_2_table

,volatility_bucket,n,declining_pages,decline_rate,median_volatility,decline_rate_pct
0,Low,19051,5702,0.299302,0.939616,29.930187
1,Medium,19050,7013,0.368136,2.294394,36.813648
2,High,19050,6636,0.348346,4.836791,34.834646
3,Very High,19050,6913,0.362887,10.008863,36.288714


In [19]:
volatility_rates = (
    signal_2_table["decline_rate"]
    .dropna()
    .to_numpy()
)

volatility_range = (
    volatility_rates.max()
    - volatility_rates.min()
)

if volatility_range < 0.03:
    signal_2_verdict = "FALSE"
elif np.all(np.diff(volatility_rates) >= 0):
    signal_2_verdict = "CONFIRMED"
elif np.all(np.diff(volatility_rates) <= 0):
    signal_2_verdict = "OPPOSITE"
else:
    signal_2_verdict = "MIXED"

print("Signal 2 verdict:", signal_2_verdict)
print(
    "Decline-rate range:",
    round(float(volatility_range), 3),
)

Signal 2 verdict: MIXED
Decline-rate range: 0.069


### Signal 2 Verdict

**Verdict: Replace this line with the printed verdict.**

Interpretation:

If decline rates rise consistently from low to very-high volatility, the signal is confirmed.

If the pattern is uneven, the verdict should be mixed. Position volatility may still be useful as supporting context, but it should not become an automatic refresh trigger.

## Signal Test 3 — Average Search Position

### Hypothesis

Pages in different position bands may show different later decline rates.

The test uses only average position measured during March 1–15.

Position is expected to provide context for visibility and CTR. However, a deeper position is not automatically evidence that content should be refreshed.

In [20]:
audit_frame["position_bucket"] = pd.cut(
    audit_frame["feature_avg_position"],
    bins=[
        -np.inf,
        3,
        10,
        20,
        np.inf,
    ],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
        "Deep",
    ],
)

signal_3_table = (
    audit_frame
    .groupby(
        "position_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_position=("feature_avg_position", "median"),
    )
    .reset_index()
)

signal_3_table["decline_rate_pct"] = (
    100 * signal_3_table["decline_rate"]
)

signal_3_table

,position_bucket,n,declining_pages,decline_rate,median_position,decline_rate_pct
0,Top 3,9140,2772,0.303282,2.242445,30.328228
1,Page 1,36709,12621,0.343812,5.630492,34.381214
2,Page 2,15079,4678,0.310233,13.637607,31.023277
3,Deep,15273,6193,0.405487,30.017759,40.548681


In [21]:
position_rates = (
    signal_3_table["decline_rate"]
    .dropna()
    .to_numpy()
)

position_range = (
    position_rates.max()
    - position_rates.min()
)

if position_range < 0.03:
    signal_3_verdict = "FALSE"
elif np.all(np.diff(position_rates) >= 0):
    signal_3_verdict = "CONFIRMED"
elif np.all(np.diff(position_rates) <= 0):
    signal_3_verdict = "OPPOSITE"
else:
    signal_3_verdict = "MIXED"

print("Signal 3 verdict:", signal_3_verdict)
print(
    "Decline-rate range:",
    round(float(position_range), 3),
)

Signal 3 verdict: MIXED
Decline-rate range: 0.102


### Signal 3 Verdict

**Verdict: Replace this line with the printed verdict.**

Interpretation:

Average position may help explain the page's search context, but it should not be interpreted as a causal ranking factor.

The result is observational and should only influence a decision-support ranking when combined with other signals.

# 3. Flag-Linked Test — CTR Relative to Position

FlyRank's CTR-review logic depends on comparing CTR with search position.

A page in position 2 naturally tends to receive a different CTR from a page in position 25. Directly comparing their raw CTR values would therefore be misleading.

For this test, I compare each page's CTR with the median CTR of other pages in the same position band.

A page is classified as below-expected CTR when its CTR is below its position band's median.

This tests whether position-adjusted low CTR is associated with a different observed decline rate.

In [22]:
# Position bands already exist from Signal Test 3

position_ctr_reference = (
    audit_frame
    .groupby(
        "position_bucket",
        observed=True,
    )["feature_ctr"]
    .median()
    .rename("expected_ctr_for_band")
)

audit_frame = audit_frame.merge(
    position_ctr_reference,
    left_on="position_bucket",
    right_index=True,
    how="left",
)

audit_frame["ctr_gap"] = (
    audit_frame["feature_ctr"]
    - audit_frame["expected_ctr_for_band"]
)

audit_frame["ctr_status"] = np.where(
    audit_frame["ctr_gap"] < 0,
    "Below band median",
    "At or above band median",
)

flag_linked_table = (
    audit_frame
    .groupby(
        [
            "position_bucket",
            "ctr_status",
        ],
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr=("feature_ctr", "median"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

flag_linked_table["decline_rate_pct"] = (
    100 * flag_linked_table["decline_rate"]
)

flag_linked_table

,position_bucket,ctr_status,n,declining_pages,decline_rate,median_ctr,median_ctr_gap,decline_rate_pct
0,Top 3,At or above band median,4570,1022,0.223632,0.005650,0.003083,22.363239
1,Top 3,Below band median,4570,1750,0.382932,0.000366,-0.002201,38.293217
2,Page 1,At or above band median,18359,5084,0.276921,0.004556,0.002731,27.692140
3,Page 1,Below band median,18350,7537,0.410736,0.000000,-0.001825,41.073569
4,Page 2,At or above band median,7541,1914,0.253812,0.004651,0.003628,25.381249
5,Page 2,Below band median,7538,2764,0.366676,0.000000,-0.001024,36.667551
6,Deep,At or above band median,15273,6193,0.405487,0.000000,0.000000,40.548681


In [23]:
flag_summary = (
    audit_frame
    .groupby(
        "ctr_status",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

flag_summary["decline_rate_pct"] = (
    100 * flag_summary["decline_rate"]
)

flag_summary

,ctr_status,n,declining_pages,decline_rate,median_ctr_gap,decline_rate_pct
0,At or above band median,45743,14213,0.310714,0.001935,31.071421
1,Below band median,30458,12051,0.395660,-0.001487,39.565960


In [24]:
flag_rate_map = dict(
    zip(
        flag_summary["ctr_status"],
        flag_summary["decline_rate"],
    )
)

below_rate = flag_rate_map.get(
    "Below band median",
    np.nan,
)

above_rate = flag_rate_map.get(
    "At or above band median",
    np.nan,
)

flag_difference = below_rate - above_rate

if pd.isna(flag_difference):
    flag_verdict = "FALSE"
elif flag_difference >= 0.03:
    flag_verdict = "CONFIRMED"
elif flag_difference <= -0.03:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("Flag-linked verdict:", flag_verdict)
print(
    "Below-minus-above decline-rate difference:",
    round(float(flag_difference), 3),
)

Flag-linked verdict: CONFIRMED
Below-minus-above decline-rate difference: 0.085


### Flag-Linked Verdict

**Verdict: Replace this line with the printed verdict.**

Interpretation:

If below-band-median CTR pages show a meaningfully higher decline rate, the CTR-versus-position assumption is supported.

If the difference is small or inconsistent, low CTR should be treated as review context rather than as proof that metadata or content is weak.

Even a confirmed result would remain observational. It would not prove that changing the title or metadata would cause recovery.

In [25]:
verdict_summary = pd.DataFrame(
    [
        {
            "signal": "Visibility / impressions",
            "verdict": signal_1_verdict,
            "role_in_baseline": (
                "Potential impact and review priority"
            ),
        },
        {
            "signal": "Position volatility",
            "verdict": signal_2_verdict,
            "role_in_baseline": (
                "Ranking-instability context"
            ),
        },
        {
            "signal": "Average position",
            "verdict": signal_3_verdict,
            "role_in_baseline": (
                "Search-position context"
            ),
        },
        {
            "signal": "CTR relative to position",
            "verdict": flag_verdict,
            "role_in_baseline": (
                "Flag-linked CTR review signal"
            ),
        },
    ]
)

verdict_summary

,signal,verdict,role_in_baseline
0,Visibility / impressions,MIXED,Potential impact and review priority
1,Position volatility,MIXED,Ranking-instability context
2,Average position,MIXED,Search-position context
3,CTR relative to position,CONFIRMED,Flag-linked CTR review signal


# 4. What This Means in Practice

The signal audit should guide the baseline rather than force a preferred conclusion.

Signals marked CONFIRMED may be used directly in the baseline rule. Signals marked MIXED may be used as supporting context with lower weight. Signals marked OPPOSITE or FALSE should not be used in the assumed direction.

A content team should treat the final score as a ranked review queue. It should not automatically edit pages, claim causal refresh impact, or claim that the tested fields prove search-engine ranking factors.

The strongest operational rule will combine potential impact with one or two evidence-backed weakness signals while keeping every recommendation open to human review.

In [26]:
print("Practical signal summary")
print("-" * 50)

for _, row in verdict_summary.iterrows():
    print(
        f"{row['signal']}: "
        f"{row['verdict']} — "
        f"{row['role_in_baseline']}"
    )

Practical signal summary
--------------------------------------------------
Visibility / impressions: MIXED — Potential impact and review priority
Position volatility: MIXED — Ranking-instability context
Average position: MIXED — Search-position context
CTR relative to position: CONFIRMED — Flag-linked CTR review signal


## Leakage and Privacy Check

The outcome proxy is used only to evaluate the signals. It is not itself a baseline input.

The later ML-07 baseline must not use:

- outcome-window impressions;
- outcome daily impressions;
- the declining proxy;
- decline ratios;
- product scores or action flags;
- raw client names, URLs, domains, titles, or queries.

Only feature-window signals may be used in the baseline score.

In [27]:
allowed_signal_columns = {
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_active_days",
    "position_bucket",
    "visibility_bucket",
    "volatility_bucket",
    "ctr_gap",
    "ctr_status",
}

forbidden_baseline_inputs = {
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "is_declining_proxy",
    "decline_ratio",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_flag",
}

overlap = sorted(
    allowed_signal_columns.intersection(
        forbidden_baseline_inputs
    )
)

print("Forbidden fields inside allowed signals:", overlap)

assert len(overlap) == 0

print("Signal-audit leakage check passed.")

Forbidden fields inside allowed signals: []
Signal-audit leakage check passed.


# Self-check

- [x] I inspected the distributions before selecting thresholds.
- [x] I noted the heavy-tail risk.
- [x] I tested three safe feature-window signals.
- [x] Every signal table shows `n`.
- [x] Every signal has a one-word verdict.
- [x] I tested a real flag-linked CTR-versus-position assumption.
- [x] I used the outcome proxy only for evaluation.
- [x] I did not use future-window or label-derived fields as baseline inputs.
- [x] I included no client names, URLs, domains, or private queries.
- [x] My conclusions use observed, measured, directional, and decision-support language.
- [x] The notebook runs from top to bottom without errors.